In [1]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.0 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.nn import GATConv
from torch_geometric.data import Data
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, precision_score
from sklearn.preprocessing import StandardScaler

# Set random seeds for reproducibility
torch.manual_seed(42)
torch.cuda.manual_seed(42)  # if using GPU
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Load and normalize datasets
data_1 = pd.read_csv('/content/Line_6s_Th.dat', sep='\\s+', header=None)
data_2 = pd.read_csv('/content/Stw_PToP_Leg.dat', sep='\\s+', header=None)

# Name the columns for clarity
data_2.columns = ['Time', 'Leg_1', 'Leg_2', 'Leg_3', 'Leg_4', 'Leg_5', 'Leg_6']
data_1.columns = ['Time', 'X', 'Y', 'Z', 'Roll', 'Pitch', 'Yaw']

# Extract input features and output labels
X = data_2[['Leg_1', 'Leg_2', 'Leg_3', 'Leg_4', 'Leg_5', 'Leg_6']].values
Y = data_1[['X', 'Y', 'Z', 'Roll', 'Pitch', 'Yaw']].values

# Normalize features and labels
scaler_X = StandardScaler()
scaler_Y = StandardScaler()
X = scaler_X.fit_transform(X)
Y = scaler_Y.fit_transform(Y)

# Split data into train and test sets with fixed random seed
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

# Convert data to torch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float)
Y_train_tensor = torch.tensor(Y_train, dtype=torch.float)
X_test_tensor = torch.tensor(X_test, dtype=torch.float)
Y_test_tensor = torch.tensor(Y_test, dtype=torch.float)

# Define a synthetic edge index for GAT, assuming fully connected nodes within the batch
def create_edge_index(num_nodes):
    return torch.combinations(torch.arange(num_nodes), r=2).t()

# Create Data objects for training and testing
train_data = Data(x=X_train_tensor, edge_index=create_edge_index(X_train_tensor.shape[0]), y=Y_train_tensor)
test_data = Data(x=X_test_tensor, edge_index=create_edge_index(X_test_tensor.shape[0]), y=Y_test_tensor)

# Define the Hybrid GAT + Regression Model
class HybridGATRegressionModel(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=128, heads=4, dropout=0.3):
        super(HybridGATRegressionModel, self).__init__()
        self.gat1 = GATConv(input_dim, hidden_dim, heads=heads, dropout=dropout)
        self.regression = nn.Sequential(
            nn.Linear(hidden_dim * heads, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.gat1(x, edge_index)
        x = torch.relu(x)
        x = self.regression(x)
        return x

# Instantiate the model, optimizer, and loss functions
input_dim = 6
output_dim = 6
model = HybridGATRegressionModel(input_dim, output_dim)
optimizer = optim.Adam(model.parameters(), lr=0.001)
mse_criterion = nn.MSELoss()
mae_criterion = nn.L1Loss()

# Training function
def train(model, data, optimizer, mse_criterion, mae_criterion, alpha=0.5):
    model.train()
    optimizer.zero_grad()
    output = model(data)
    mse_loss = mse_criterion(output, data.y)
    mae_loss = mae_criterion(output, data.y)
    loss = alpha * mse_loss + (1 - alpha) * mae_loss
    loss.backward()
    optimizer.step()
    return loss.item()

# Evaluation function with overall metrics
def evaluate(model, data):
    model.eval()
    with torch.no_grad():
        output = model(data).cpu()
        target = data.y.cpu()

        # Metrics for each leg and output
        metrics = {f'Leg_{i+1}': {} for i in range(6)}
        output_labels = ['X', 'Y', 'Z', 'Roll', 'Pitch', 'Yaw']
        leg_labels = [f'Leg_{i+1}' for i in range(6)]

        # Overall metrics accumulators
        all_target_bin = []
        all_output_bin = []

        for leg_index, leg_label in enumerate(leg_labels):
            for output_index, output_label in enumerate(output_labels):
                mse = mean_squared_error(target[:, output_index], output[:, output_index])
                mae = mean_absolute_error(target[:, output_index], output[:, output_index])
                r2 = r2_score(target[:, output_index], output[:, output_index])

                # Calculate accuracy and precision (assumes rounding outputs for discrete evaluation)
                target_bin = (target[:, output_index].numpy() > 0).astype(int)
                output_bin = (output[:, output_index].numpy() > 0).astype(int)

                # Accumulate for overall metrics
                all_target_bin.extend(target_bin)
                all_output_bin.extend(output_bin)

                accuracy = accuracy_score(target_bin, output_bin)
                precision = precision_score(target_bin, output_bin, zero_division=0)

                metrics[leg_label][output_label] = {
                    'MSE': mse,
                    'MAE': mae,
                    'R²': r2,
                    'Accuracy': accuracy,
                    'Precision': precision
                }

        # Calculate overall metrics
        overall_accuracy = accuracy_score(all_target_bin, all_output_bin)
        overall_precision = precision_score(all_target_bin, all_output_bin, zero_division=0)
        overall_mse = mean_squared_error(target, output)
        overall_mae = mean_absolute_error(target, output)
        overall_r2 = r2_score(target, output)

    return metrics, overall_mse, overall_mae, overall_r2, overall_accuracy, overall_precision

# Training loop
epochs = 300
best_mae = float('inf')
for epoch in range(epochs):
    train_loss = train(model, train_data, optimizer, mse_criterion, mae_criterion, alpha=0.5)

    if epoch % 10 == 0 or epoch == epochs - 1:
        metrics, overall_mse, overall_mae, overall_r2, overall_accuracy, overall_precision = evaluate(model, test_data)

        # Save the model if MAE improves
        if overall_mae < best_mae:
            best_mae = overall_mae
            torch.save(model.state_dict(), 'best_model_hybrid.pth')

        print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, "
              f"Test MAE = {overall_mae:.4f}, Test R² = {overall_r2:.4f}, "
              f"Accuracy = {overall_accuracy:.4f}, Precision = {overall_precision:.4f}")

# Load best model and final evaluation
model.load_state_dict(torch.load('best_model_hybrid.pth'))
metrics, overall_mse, overall_mae, overall_r2, overall_accuracy, overall_precision = evaluate(model, test_data)

# Print final metrics
print("Final Test Metrics:")
print("Individual Leg and Motion Metrics:")
for leg_label, leg_metrics in metrics.items():
    print(f"{leg_label} metrics:")
    for output_label, output_metrics in leg_metrics.items():
        print(f"  {output_label}: MSE={output_metrics['MSE']:.4f}, "
              f"MAE={output_metrics['MAE']:.4f}, R²={output_metrics['R²']:.4f}, "
              f"Accuracy={output_metrics['Accuracy']:.4f}, Precision={output_metrics['Precision']:.4f}")

print(f"Overall Metrics - MSE: {overall_mse:.4f}, MAE: {overall_mae:.4f}, "
      f"R² Score: {overall_r2:.4f}, Accuracy: {overall_accuracy:.4f}, Precision: {overall_precision:.4f}")


Epoch 0: Train Loss = 0.9156, Test MAE = 0.9644, Test R² = -0.0156, Accuracy = 0.4570, Precision = 0.4318
Epoch 10: Train Loss = 0.8094, Test MAE = 0.8723, Test R² = 0.1599, Accuracy = 0.6989, Precision = 0.6116
Epoch 20: Train Loss = 0.5482, Test MAE = 0.6629, Test R² = 0.4946, Accuracy = 0.9032, Precision = 0.8421
Epoch 30: Train Loss = 0.3792, Test MAE = 0.3883, Test R² = 0.7517, Accuracy = 0.9355, Precision = 0.8901
Epoch 40: Train Loss = 0.2967, Test MAE = 0.3590, Test R² = 0.8210, Accuracy = 0.9194, Precision = 0.8617
Epoch 50: Train Loss = 0.2835, Test MAE = 0.3214, Test R² = 0.8405, Accuracy = 0.9355, Precision = 0.8901
Epoch 60: Train Loss = 0.2272, Test MAE = 0.3303, Test R² = 0.8411, Accuracy = 0.9355, Precision = 0.8901
Epoch 70: Train Loss = 0.2896, Test MAE = 0.3242, Test R² = 0.8485, Accuracy = 0.9355, Precision = 0.8901
Epoch 80: Train Loss = 0.2729, Test MAE = 0.3094, Test R² = 0.8542, Accuracy = 0.9355, Precision = 0.8901
Epoch 90: Train Loss = 0.2627, Test MAE = 0.30

<ipython-input-2-0e03845d6abd>:160: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model_hybrid.pth'))
